# LeetCode #1143 — Longest Common Subsequence: Dãy con chung dài nhất

**Medium · Python 3 · String · 2D Dynamic Programming**  
[Đề gốc](https://leetcode.com/problems/longest-common-subsequence/) · [Đối chiếu mô tả](https://leetcode.doocs.org/en/lc/1143/)

```python
class Solution:
    def longestCommonSubsequence(self, text1: str, text2: str) -> int:
        ...
```

**Câu hỏi xuyên suốt:** Vì sao bài này không thể chỉ so vị trí giống nhau, nhưng lại có thể giải bằng một bảng chứa câu trả lời cho những cặp **tiền tố ngắn hơn**?

## 1. Mục tiêu bài học

1. Phân biệt rõ **subsequence (dãy con không cần liên tiếp)** và **substring (chuỗi con liên tiếp)**.
2. Hiểu chọn giữ / bỏ ký tự trong hai chuỗi.
3. Biết viết đệ quy trực giác, nhận ra trạng thái lặp.
4. Xây bảng `dp[i][j]` với hàng/cột rỗng để không vượt biên.
5. Truy vết bảng để khôi phục **một** dãy LCS và tối ưu bộ nhớ xuống `O(min(m,n))`.
6. Viết code nộp LeetCode, test chéo và hoàn thành 5 bài tập.

## 2. Đề bài — diễn giải sát nghĩa

Cho hai chuỗi `text1` và `text2`. Một **subsequence** được tạo bằng cách xóa **một số ký tự hoặc không xóa gì**, nhưng **giữ nguyên thứ tự tương đối** các ký tự còn lại.

Tìm **độ dài** của dãy con dài nhất xuất hiện trong **cả hai** chuỗi. Nếu không có ký tự nào chung, trả `0`.

| `text1` | `text2` | Đáp án | Giải thích |
|---|---|---:|---|
| `"abcde"` | `"ace"` | 3 | Một LCS là `"ace"` |
| `"abc"` | `"abc"` | 3 | Hai chuỗi bằng nhau |
| `"abc"` | `"def"` | 0 | Không có ký tự chung |

**Constraints:** `1 <= len(text1), len(text2) <= 1000`, các ký tự là chữ thường `a..z`.

In [1]:
s='abcde'
indices=[0,2,4]
subseq=''.join(s[i] for i in indices)
print('Chuỗi gốc:',s,'| lấy các index:',indices,'->',subseq)
assert subseq=='ace'

Chuỗi gốc: abcde | lấy các index: [0, 2, 4] -> ace


## 3. Subsequence khác substring thế nào?

- **Substring:** ký tự **liên tiếp**. Trong `"abcde"`, `"bcd"` là substring.
- **Subsequence:** có thể **bỏ qua** ký tự, nhưng không được đảo thứ tự. Trong `"abcde"`, `"ace"` là subsequence nhưng **không** là substring.
- `"eca"` không là subsequence của `"abcde"` vì thứ tự đảo ngược.

Bài #1143 hỏi **subsequence**, không phải chuỗi con liên tiếp. Dùng thuật toán tìm *longest common substring* sẽ trả sai nhiều trường hợp.

In [2]:
def is_subsequence(small, large):
    j=0
    for ch in large:
        if j<len(small) and ch==small[j]:
            j+=1
    return j==len(small)

for candidate in ['ace','bcd','aec','eca','']:
    print(candidate or '(rỗng)', 'là subsequence?', is_subsequence(candidate,'abcde'))
assert is_subsequence('ace','abcde')
assert not is_subsequence('eca','abcde')

ace là subsequence? True
bcd là subsequence? True
aec là subsequence? False
eca là subsequence? False
(rỗng) là subsequence? True


## 4. Python cơ bản: `len`, index, `join` và slicing

- `len(s)` trả số ký tự.
- `s[i]` trả ký tự thứ `i`; Python đánh chỉ số từ 0.
- `s[i:]` là hậu tố bắt đầu từ `i`, dùng khi minh họa đệ quy.
- `''.join(list_of_chars)` ghép các ký tự thành một chuỗi.

Nếu muốn so ký tự ở vị trí thứ `i` theo kiểu **đếm từ 1**, trong Python phải truy cập `s[i-1]`.

In [3]:
s='abc'
print('len:',len(s),'| s[0]:',s[0],'| s[1:]:',s[1:])
print('Nối ký tự:', ''.join(['a','c']))
assert s[2]=='c' and s[1:]=='bc'

len: 3 | s[0]: a | s[1:]: bc
Nối ký tự: ac


## 5. Cách trực giác: chọn hoặc bỏ ký tự

Đặt `f(i,j)` là chiều dài LCS của **hai hậu tố** `text1[i:]` và `text2[j:]`.

- Nếu một hậu tố đã hết, đáp án là 0.
- Nếu `text1[i] == text2[j]`, ta có thể lấy ký tự chung này rồi giải hai hậu tố tiếp theo: `1 + f(i+1,j+1)`.
- Nếu khác nhau, hai ký tự đầu không thể đồng thời góp mặt trong cùng một LCS; phải bỏ **ít nhất một trong hai**: `max(f(i+1,j), f(i,j+1))`.

Ta vừa thu được lời giải đệ quy đúng, nhưng chưa tối ưu.

In [4]:
def lcs_bruteforce(text1,text2):
    m,n=len(text1),len(text2)
    def dfs(i,j):
        if i==m or j==n: return 0
        if text1[i]==text2[j]:
            return 1+dfs(i+1,j+1)
        return max(dfs(i+1,j),dfs(i,j+1))
    return dfs(0,0)

for a,b in [('abcde','ace'),('abc','def'),('abc','abc'),('ab','ba')]:
    print(a,b,'->',lcs_bruteforce(a,b))
assert lcs_bruteforce('abcde','ace')==3

abcde ace -> 3
abc def -> 0
abc abc -> 3
ab ba -> 1


## 6. Nút thắt: một cặp hậu tố bị tính nhiều lần

Ví dụ `text1="abca"`, `text2="baca"`: các nhánh *bỏ ký tự ở chuỗi 1* và *bỏ ký tự ở chuỗi 2* có thể dẫn đến **cùng cặp `(i,j)`**. Đệ quy thuần sẽ tính lại.

**Dynamic Programming (quy hoạch động):** lưu đáp án bài toán nhỏ để dùng lại. Với LCS có nhiều nhất `(m+1)*(n+1)` cặp chỉ số. Đây là lý do có thể tránh sự bùng nổ nhánh đệ quy.

## 7. Top-down: thêm `@cache` để không giải lại

**`@cache`** nhớ kết quả theo cặp `(i,j)`. Mỗi trạng thái được tính một lần, công thức không thay đổi. Tuy nhiên, **đừng nộp bản đệ quy này khi hai chuỗi dài tới 1000**: đường gọi sâu có thể vượt giới hạn đệ quy mặc định của Python. Ta sẽ chuyển sang bảng lặp **bottom-up** để nộp.

In [5]:
from functools import cache

def lcs_memo(text1,text2):
    m,n=len(text1),len(text2)
    @cache
    def dfs(i,j):
        if i==m or j==n: return 0
        if text1[i]==text2[j]:
            return 1+dfs(i+1,j+1)
        return max(dfs(i+1,j),dfs(i,j+1))
    return dfs(0,0)

assert lcs_memo('abcde','ace')==3
assert lcs_memo('abc','def')==0
print('Top-down (chuỗi nhỏ): PASS')

Top-down (chuỗi nhỏ): PASS


## 8. Bottom-up — đổi ý nghĩa trạng thái sang **tiền tố**

Đặt `dp[i][j]` = độ dài LCS của **`i` ký tự đầu `text1`** và **`j` ký tự đầu `text2`**.

- `dp[0][j]=0`: chuỗi thứ nhất rỗng.
- `dp[i][0]=0`: chuỗi thứ hai rỗng.
- Đáp án nằm ở `dp[m][n]`.

Nhớ: **`dp[i][j]` xét `i` ký tự**, nên ký tự cuối của tiền tố đó là **`text1[i-1]`**, không phải `text1[i]`.

## 9. Công thức chuyển trạng thái LCS

Nếu hai ký tự cuối **bằng nhau**:

\[dp[i][j] = dp[i-1][j-1]+1.\]

Nếu **khác nhau**:

\[dp[i][j] = \max\big(dp[i-1][j],\ dp[i][j-1]\big).\]

Điều này có nghĩa: khi không thể ghép hai ký tự cuối thành một cặp, ta chọn bỏ ký tự cuối **ở một trong hai chuỗi**. Không lấy `dp[i-1][j-1]` trong nhánh khác nhau vì như thế có thể bỏ **cả hai** một cách không cần thiết.

In [6]:
# Minh họa 1 phép chuyển: tiền tố 'ab' và 'ac'
a,b='ab','ac'
print('Ký tự cuối:',a[-1],b[-1],'-> khác nhau')
print('Chọn tốt hơn trong LCS("a","ac") và LCS("ab","a")')
assert lcs_bruteforce('a','ac') == 1
assert lcs_bruteforce('ab','a') == 1

Ký tự cuối: b c -> khác nhau
Chọn tốt hơn trong LCS("a","ac") và LCS("ab","a")


## 10. Python nền tảng: tạo bảng 2 chiều

`dp = [[0]*(n+1) for _ in range(m+1)]` tạo `m+1` **hàng độc lập**, mỗi hàng `n+1` số 0.

**Không dùng** `[[0]*(n+1)]*(m+1)`: tất cả các hàng đều tham chiếu **cùng một list**; sửa một ô sẽ làm nhiều hàng đổi theo.

In [7]:
correct=[[0]*3 for _ in range(2)]
correct[0][1]=7
wrong=[[0]*3]*2
wrong[0][1]=7
print('Đúng:',correct)
print('Sai:',wrong)
assert correct==[[0,7,0],[0,0,0]]
assert wrong==[[0,7,0],[0,7,0]]

Đúng: [[0, 7, 0], [0, 0, 0]]
Sai: [[0, 7, 0], [0, 7, 0]]


## 11. Giải đầy đủ bằng bảng `dp`

Duyệt `i=1..m`, `j=1..n`. Ô `dp[i][j]` chỉ đọc các ô **ở hàng trên** hoặc **bên trái**, đều đã được tính. Đó là lý do thứ tự duyệt từ trên xuống, trái sang phải hợp lệ.

Mã demo dùng tên `lcs_dp_table` riêng để có thể quan sát cả bảng.

In [8]:
def lcs_dp_table(text1,text2):
    m,n=len(text1),len(text2)
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            if text1[i-1]==text2[j-1]:
                dp[i][j]=dp[i-1][j-1]+1
            else:
                dp[i][j]=max(dp[i-1][j],dp[i][j-1])
    return dp

dp_demo=lcs_dp_table('abc','ac')
for row in dp_demo:
    print(row)
assert dp_demo==[[0,0,0],[0,1,1],[0,1,1],[0,1,2]]

[0, 0, 0]
[0, 1, 1]
[0, 1, 1]
[0, 1, 2]


## 12. Trace bằng tay: `text1="abc"`, `text2="ac"`

| Tiền tố text1 \ text2 | `""` | `"a"` | `"ac"` |
|---|---:|---:|---:|
| `""` | 0 | 0 | 0 |
| `"a"` | 0 | **1** | 1 |
| `"ab"` | 0 | 1 | **1** |
| `"abc"` | 0 | 1 | **2** |

- `dp[1][1]=1` vì `a==a`.
- `dp[2][2]=max(dp[1][2],dp[2][1])=1` vì `b!=c`.
- `dp[3][2]=dp[2][1]+1=2` vì `c==c`.

Mỗi số không phải chỉ là một phép đếm trừu tượng: nó là **độ dài LCS tốt nhất của hai tiền tố** tương ứng.

In [9]:
def trace_lcs_table(a,b):
    dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            if a[i-1]==b[j-1]:
                dp[i][j]=dp[i-1][j-1]+1
                reason='khớp -> chéo + 1'
            else:
                dp[i][j]=max(dp[i-1][j],dp[i][j-1])
                reason='khác -> max(trên, trái)'
            print(f'i={i},j={j}, ký tự={a[i-1]}/{b[j-1]}, '
                  f'dp[{i}][{j}]={dp[i][j]} ({reason})')
    print('Bảng cuối:')
    for row in dp: print(row)
    return dp[-1][-1]

assert trace_lcs_table('abc','ac')==2

i=1,j=1, ký tự=a/a, dp[1][1]=1 (khớp -> chéo + 1)
i=1,j=2, ký tự=a/c, dp[1][2]=1 (khác -> max(trên, trái))
i=2,j=1, ký tự=b/a, dp[2][1]=1 (khác -> max(trên, trái))
i=2,j=2, ký tự=b/c, dp[2][2]=1 (khác -> max(trên, trái))
i=3,j=1, ký tự=c/a, dp[3][1]=1 (khác -> max(trên, trái))
i=3,j=2, ký tự=c/c, dp[3][2]=2 (khớp -> chéo + 1)
Bảng cuối:
[0, 0, 0]
[0, 1, 1]
[0, 1, 1]
[0, 1, 2]


## 13. Nối từ demo sang code nộp LeetCode

Bản `class Solution` cần **trả một số nguyên**. Không trả cả bảng, không trả chuỗi LCS. Bảng trong method tự tạo và tự sử dụng, nên không cần chạy cell khác mới hoạt động.

In [10]:
class Solution:
    def longestCommonSubsequence(self, text1: str, text2: str) -> int:
        m,n=len(text1),len(text2)
        dp=[[0]*(n+1) for _ in range(m+1)]
        for i in range(1,m+1):
            for j in range(1,n+1):
                if text1[i-1]==text2[j-1]:
                    dp[i][j]=dp[i-1][j-1]+1
                else:
                    dp[i][j]=max(dp[i-1][j],dp[i][j-1])
        return dp[m][n]

sol=Solution()
assert sol.longestCommonSubsequence('abcde','ace')==3
assert sol.longestCommonSubsequence('abc','abc')==3
assert sol.longestCommonSubsequence('abc','def')==0
print('LeetCode Solution: PASS')

LeetCode Solution: PASS


## 14. Vì sao DP đúng? Chứng minh ngắn

Quy nạp theo tổng độ dài tiền tố `i+j`:

- Nếu một tiền tố rỗng, LCS có độ dài 0: bảng đúng.
- Nếu hai ký tự cuối **trùng nhau**, tồn tại một LCS dùng cặp ký tự cuối đó; khi bỏ cặp ấy, phần trước tối ưu có độ dài `dp[i-1][j-1]`.
- Nếu **khác nhau**, một LCS không thể đồng thời chứa **cả hai** ký tự cuối như cùng vị trí ghép; ít nhất một ký tự cuối bị bỏ. Do đó chọn tốt nhất giữa hai tiền tố bỏ một ký tự.

Mọi ô phụ thuộc vào ô của các tiền tố **ngắn hơn**, đã được tính đúng, nên `dp[m][n]` đúng.

## 15. Độ phức tạp: hai kích thước độc lập

Gọi `m=len(text1)`, `n=len(text2)`.

- Bảng có `(m+1)*(n+1)` ô ⇒ **O(m*n)** thời gian (tính một lần/ô).
- Bộ nhớ bảng **O(m*n)**.
- Có thể giảm xuống **O(min(m,n))** bộ nhớ vì mỗi ô chỉ đọc hàng hiện tại + hàng trước. Ta học riêng phương án đó bên dưới.

Lưu ý: `m,n` không nhất thiết bằng nhau; không ghi chung `O(n²)` nếu chưa định nghĩa `n`.

## 16. Biến thể hữu ích: khôi phục một chuỗi LCS

Đề chỉ hỏi **độ dài**, nhưng có thể học thêm cách **backtrack**: đi ngược từ ô `(m,n)`.

- Nếu ký tự cuối giống nhau, đó là một ký tự của LCS ⇒ lưu lại rồi đi chéo lên trái.
- Nếu khác nhau, đi đến ô có giá trị lớn hơn giữa **trên** và **trái**. Nếu bằng nhau, chọn một hướng tùy ý.

Kết quả được tìm theo thứ tự **ngược**, nên phải đảo trước khi nối. **LCS có thể không duy nhất.**

In [11]:
def reconstruct_one_lcs(a,b):
    dp=lcs_dp_table(a,b)
    i,j=len(a),len(b)
    chars=[]
    while i>0 and j>0:
        if a[i-1]==b[j-1]:
            chars.append(a[i-1])
            i-=1
            j-=1
        elif dp[i-1][j]>=dp[i][j-1]:
            i-=1
        else:
            j-=1
    return ''.join(reversed(chars))

s=reconstruct_one_lcs('abcde','ace')
print('Một dãy LCS:',s)
assert s=='ace'
s2=reconstruct_one_lcs('abc','bac')
assert len(s2)==2 and is_subsequence(s2,'abc') and is_subsequence(s2,'bac')

Một dãy LCS: ace


## 17. Tối ưu bộ nhớ: mỗi lần chỉ cần một hàng

Khi cập nhật ô `dp[i][j]`, ta cần:

- `dp[i-1][j]`: ô **trên** (trong hàng `prev`).
- `dp[i][j-1]`: ô **trái** (đã có trong hàng `curr`).
- `dp[i-1][j-1]`: ô **chéo trên trái** (trong hàng `prev`).

Vì vậy đủ dùng hai mảng `prev`, `curr` độ dài `n+1`; hết một hàng thì `prev=curr`. Để giảm bộ nhớ thêm, đặt chuỗi ngắn hơn vào **chiều cột**.

In [12]:
def lcs_two_rows(a,b):
    if len(b)>len(a):
        a,b=b,a
    prev=[0]*(len(b)+1)
    for ch_a in a:
        curr=[0]*(len(b)+1)
        for j,ch_b in enumerate(b,1):
            if ch_a==ch_b:
                curr[j]=prev[j-1]+1
            else:
                curr[j]=max(prev[j],curr[j-1])
        prev=curr
    return prev[-1]

for a,b,expected in [('abcde','ace',3),('abc','def',0),('abc','abc',3),('aaaa','aa',2)]:
    assert lcs_two_rows(a,b)==expected
print('Tối ưu bộ nhớ: PASS')

Tối ưu bộ nhớ: PASS


## 18. Trace hai hàng: vì sao không ghi đè mất dữ liệu?

Ta tạo `curr` **mới** ở mỗi vòng lặp chuỗi `a`, nên `prev[j-1]` vẫn thuộc **hàng trước**, không phải kết quả mới ghi đè. Nếu dùng `prev=curr` rồi sửa `curr` trực tiếp, phải rất cẩn thận với vấn đề alias (hai biến cùng trỏ vào một list).

In [13]:
def trace_two_rows(a,b):
    prev=[0]*(len(b)+1)
    print('ban đầu:  ',prev)
    for ca in a:
        curr=[0]*(len(b)+1)
        for j,cb in enumerate(b,1):
            curr[j] = prev[j-1]+1 if ca==cb else max(prev[j],curr[j-1])
        print(f'sau {ca!r}:',curr)
        prev=curr
    return prev[-1]

assert trace_two_rows('abc','ac')==2

ban đầu:   [0, 0, 0]
sau 'a': [0, 1, 1]
sau 'b': [0, 1, 1]
sau 'c': [0, 1, 2]


## 19. Test mẫu, đối xứng, lặp ký tự và biên

`LCS(a,b) = LCS(b,a)` vì định nghĩa dãy con chung đối xứng. Nhưng **kết quả chuỗi cụ thể** của hàm truy vết có thể khác khi đổi thứ tự — đề gốc chỉ kiểm tra **độ dài**.

Một ký tự đơn, tất cả ký tự giống nhau, không có ký tự chung, và các ký tự lặp là những ca cần kiểm thử.

In [14]:
cases=[
    ('a','a',1),('a','b',0),('aaaa','aa',2),
    ('abcde','ace',3),('abc','def',0),('abc','abc',3),
    ('abc','bac',2),('axbycz','abc',3),('zzxyz','zyz',3)
]
for a,b,expected in cases:
    x=Solution().longestCommonSubsequence(a,b)
    y=lcs_two_rows(a,b)
    assert x==y==expected,(a,b,x,y,expected)
    assert Solution().longestCommonSubsequence(b,a)==expected
print('PASS',len(cases),'ca tiêu biểu và tính đối xứng')

PASS 9 ca tiêu biểu và tính đối xứng


## 20. Kiểm thử đối chiếu ngẫu nhiên trên input nhỏ

Ta dùng **ba cách**: đệ quy không cache (oracle trên input rất ngắn), DP bảng, DP hai hàng. So hàng trăm cặp chuỗi ngắn để phát hiện lỗi `i-1`, nhánh match/mismatch, hoặc thứ tự cập nhật.

In [15]:
import random
rng=random.Random(1143)
for _ in range(250):
    a=''.join(rng.choice('abc') for _ in range(rng.randint(1,6)))
    b=''.join(rng.choice('abc') for _ in range(rng.randint(1,6)))
    expected=lcs_bruteforce(a,b)
    assert Solution().longestCommonSubsequence(a,b)==expected,(a,b)
    assert lcs_two_rows(a,b)==expected,(a,b)
    reconstructed=reconstruct_one_lcs(a,b)
    assert len(reconstructed)==expected
    assert is_subsequence(reconstructed,a) and is_subsequence(reconstructed,b)
print('PASS 250 cặp chuỗi + khôi phục LCS')

PASS 250 cặp chuỗi + khôi phục LCS


## 21. Test giới hạn độ dài `1000`

Đây là lý do chọn **bottom-up** cho bản submit. Với hai chuỗi giống hệt 1000 ký tự, đáp án luôn bằng 1000. Với hai chuỗi hoàn toàn khác ký tự, đáp án luôn bằng 0.

In [16]:
a='a'*1000
b='b'*1000
assert Solution().longestCommonSubsequence(a,a)==1000
assert Solution().longestCommonSubsequence(a,b)==0
assert lcs_two_rows(a,a)==1000
print('PASS chuỗi độ dài 1000')

PASS chuỗi độ dài 1000


## 22. Lỗi thường gặp

| Lỗi | Tại sao | Sửa |
|---|---|---|
| Chỉ xét đoạn liên tiếp | Nhầm substring và subsequence | Dùng công thức LCS |
| `text1[i]` trong vòng `i=1..m` | Lệch index và có thể vượt biên | `text1[i-1]` |
| `dp=[[0]*n]*m` | Các hàng cùng tham chiếu | List comprehension |
| Hai ký tự khác nhau ⇒ đi chéo | Bỏ cùng lúc cả hai ký tự | `max(trên,trái)` |
| Dùng đệ quy với chuỗi dài 1000 | Có thể vượt recursion limit | Bottom-up |
| Trả chuỗi thay vì số | Sai yêu cầu LeetCode | `return dp[m][n]` |

## 23. Pattern và biến thể

Gợi ý dùng DP 2 chiều khi: có **hai chuỗi/dãy**, cần so thứ tự, và lựa chọn cốt lõi là **giữ/bỏ phần tử cuối** hoặc **khớp/không khớp**.

LCS liên hệ với bài **Edit Distance** nhưng công thức khác: Edit Distance đếm số thao tác sửa/chèn/xóa; LCS đếm số ký tự được giữ **chung**. Đừng hoán đổi hai bảng DP chỉ vì cùng có kích thước `(m+1)*(n+1)`.

## 24. Năm bài tập tự làm

1. **Tính tay:** lập bảng DP cho `"ab"` và `"ac"`; đáp án là bao nhiêu?
2. **Đoán output:** `is_subsequence("ac","abc")` và `is_subsequence("ca","abc")` lần lượt là gì?
3. **TODO:** viết `exercise_lcs_table(a,b)` trả về **toàn bảng**, có hàng/cột rỗng.
4. **TODO:** viết `exercise_lcs_rolling(a,b)` dùng hai hàng, chỉ `O(min(m,n))` bộ nhớ.
5. **Mini challenge:** `exercise_reconstruct(a,b)` trả về **một** chuỗi LCS (không bắt buộc duy nhất); kiểm bằng độ dài và `is_subsequence`.

In [17]:
# BÀI TẬP — TODO chưa cài, không chạy tự động.
def exercise_lcs_table(a,b):
    # TODO 1: tạo bảng (len(a)+1) x (len(b)+1)
    # TODO 2: xử lý match/mismatch; return bảng
    pass

def exercise_lcs_rolling(a,b):
    # TODO 1: đặt chuỗi ngắn vào b
    # TODO 2: vòng lặp prev/curr; return độ dài LCS
    pass

def exercise_reconstruct(a,b):
    # TODO 1: lấy bảng DP của bài 3
    # TODO 2: đi ngược từ ô cuối; đảo list ký tự
    pass

# Tự kiểm tra sau khi hoàn thành:
# assert exercise_lcs_table('abc','ac')[-1][-1] == 2
# assert exercise_lcs_rolling('abcde','ace') == 3
# r = exercise_reconstruct('abc','bac')
# assert len(r) == 2 and is_subsequence(r,'abc') and is_subsequence(r,'bac')

## 25. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Một ký tự khớp | `dp[1][1]=1` | `dp[2][2]=1` |
| 2 | Giữ thứ tự | Quét từ trái sang phải | `True`, `False` |
| 3 | Hàng/cột rỗng đều 0 | `range(1,m+1)` | Match = chéo + 1; khác = max trên/trái |
| 4 | Chỉ dùng hàng liền trước | `prev`, `curr` | `prev=curr` sau mỗi hàng |
| 5 | Bắt đầu ở góc dưới phải | Trùng thì đi chéo | Dùng `reversed(chars)` |

## 26. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** Bảng cho `"ab"/"ac"` có hàng lần lượt `[0,0,0]`, `[0,1,1]`, `[0,1,1]`, nên đáp án **1**.  
**Bài 2:** `True`, `False`.

**Bài 3–5:** các hàm tham khảo được kiểm tra bằng assert bên dưới. Một LCS có thể có **nhiều đáp án chuỗi**, vì thế với bài 5 không nên so bằng một chuỗi cố định.

In [18]:
def answer_lcs_table(a,b):
    dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]+1
            else: dp[i][j]=max(dp[i-1][j],dp[i][j-1])
    return dp

def answer_lcs_rolling(a,b):
    if len(b)>len(a): a,b=b,a
    prev=[0]*(len(b)+1)
    for ch in a:
        curr=[0]*(len(b)+1)
        for j,x in enumerate(b,1):
            curr[j]=prev[j-1]+1 if ch==x else max(prev[j],curr[j-1])
        prev=curr
    return prev[-1]

def answer_reconstruct(a,b):
    dp=answer_lcs_table(a,b)
    i,j=len(a),len(b)
    out=[]
    while i and j:
        if a[i-1]==b[j-1]:
            out.append(a[i-1]); i-=1; j-=1
        elif dp[i-1][j]>=dp[i][j-1]: i-=1
        else: j-=1
    return ''.join(reversed(out))

assert answer_lcs_table('abc','ac')[-1][-1]==2
assert answer_lcs_rolling('abcde','ace')==3
r=answer_reconstruct('abc','bac')
assert len(r)==2 and is_subsequence(r,'abc') and is_subsequence(r,'bac')
print('Đáp án tham khảo: PASS')

Đáp án tham khảo: PASS


## 27. Template nhớ nhanh & checklist submit

```text
dp[i][j] = LCS của i ký tự đầu text1, j ký tự đầu text2
i=0 hoặc j=0 -> 0
Nếu text1[i-1] == text2[j-1]: dp[i-1][j-1] + 1
Nếu khác: max(dp[i-1][j], dp[i][j-1])
Đáp án: dp[m][n]
```

- [ ] Subsequence có thể **không liên tiếp** nhưng giữ thứ tự.
- [ ] Bảng có `m+1` hàng, `n+1` cột; không sai `i-1`.
- [ ] Không dùng `[[0]*n]*m`.
- [ ] Code submit trả **int**, không trả dãy con.
- [ ] Test chuỗi giống nhau, không có ký tự chung, ký tự lặp, độ dài 1000.